# Module 7 · Solutions

In [ ]:
import numpy as np, pandas as pd
from scipy.optimize import linprog
yields = np.array([0.062, 0.068, 0.071, 0.084])
names = ["Overnight","T-bills","Bank FD","Corp"]

## 7A

In [ ]:
# Ex1 - overnight buffer x1 >= 10  (as -x1 <= -10)
def solve(A_extra=None, b_extra=None, y=yields):
    A = [[-1,-1,0,0],[0,0,0,1],[0,0,1,0]]; b = [-30,20,35]
    if A_extra: A += A_extra; b += b_extra
    r = linprog(-y, A_ub=A, b_ub=b, A_eq=[[1,1,1,1]], b_eq=[100], bounds=[(0,None)]*4, method="highs")
    return r
base = solve()
buf  = solve([[-1,0,0,0]], [-10])
print("With buffer:", pd.Series(buf.x, index=names).round(1).to_dict())
print(f"Cost of the buffer: Rs {(-base.fun) - (-buf.fun):.3f} cr/yr")
print("T-bills paid for it: the Rs 10 cr moved from 6.8% T-bills to 6.2% Overnight -> cost = 10 x 0.6% = 0.06 cr. Priced.")

# Ex2 - corp yield falls to 7.0%
y2 = yields.copy(); y2[3] = 0.070
r2 = solve(y=y2)
print("\nCorp at 7.0%:", pd.Series(r2.x, index=names).round(1).to_dict())
print("Corp cap no longer binds - the solver ABANDONS corp entirely for the FD at 7.1%.")
print("Lesson: a 1.4pp input change flipped an allocation from 'max allowed' to 'zero'.")
print("Optimal VALUES move smoothly; optimal ALLOCATIONS jump. Optimisers are drama queens about inputs -")
print("carry this straight into Module 12, where the inputs are estimated returns.")

## 7B

In [ ]:
# Ex1 - salvage rises to 90
p = 0.55; PAY, COST, PILOT = 300, 120, 15
for SALV in [40, 90]:
    launch = p*(PAY-COST) + (1-p)*(SALV-COST)
    pilot  = p*(PAY-COST) - PILOT
    be = 1 + PILOT/(SALV-COST)
    print(f"salvage {SALV}: launch EMV {launch:+.1f} | pilot EMV {pilot:+.1f} | launch overtakes pilot above p={be:.0%}")
print("\nBetter salvage shrinks the DOWNSIDE that the pilot exists to avoid. The pilot's whole value")
print("is the right to walk away from a bad branch; make the bad branch less bad and that right is worth less.")

In [ ]:
# Ex2 - loss-averse board: losses weighted 2x
def emv_averse(outcomes):   # list of (prob, value)
    return sum(pr * (v*2 if v < 0 else v) for pr, v in outcomes)
launch = emv_averse([(0.55, 300-120), (0.45, 40-120)])
pilot  = emv_averse([(0.55, 300-120-15), (0.45, -15)])
print(f"Loss-averse: launch {launch:+.1f} | pilot {pilot:+.1f} | none 0")
print("Pilot still wins, and by MORE - its worst case is only -15, barely touched by the 2x penalty.")
print("A recommendation that ranks first under EMV *and* under loss-aversion is robust to the room's")
print("risk appetite - which means you can present it without first winning an argument about utility.")

In [ ]:
# Ex3 - per-scenario optima
scen = pd.DataFrame({
    "prob":[.30,.50,.20],
    "Overnight":[.054,.062,.072], "T-bills":[.060,.068,.078],
    "Bank FD":[.068,.071,.073], "Corp":[.080,.084,.086]}, index=["fall","base","rise"])
for s in scen.index:
    y = scen.loc[s, ["Overnight","T-bills","Bank FD","Corp"]].values.astype(float)
    r = linprog(-y, A_ub=[[-1,-1,0,0],[0,0,0,1],[0,0,1,0]], b_ub=[-30,20,35],
                A_eq=[[1,1,1,1]], b_eq=[100], bounds=[(0,None)]*4, method="highs")
    print(s, ":", dict(zip(["ON","TB","FD","Corp"], np.round(r.x,1))))
print("\nAll three worlds produce essentially the SAME allocation here (caps bind regardless) -")
print("so the weighted answer can be presented with confidence. When per-scenario optima DIVERGE,")
print("that divergence is exactly the uncertainty your presentation must carry.")